In [ ]:
import sys
sys.path.append("..")

from src.spark_utils import load_config, get_spark, read_raw_csv, prepare_table, write_to_mysql
from src.table_configs import TABLES, LOAD_ORDER, rename_map, type_map

In [ ]:
cfg = load_config()
spark = get_spark(cfg["jdbc_jar"])

In [ ]:
for table_key in LOAD_ORDER:
    table_cfg = TABLES[table_key]
    file_path = cfg["raw_dir"] / table_cfg["file"]

    print(f"Loading: {table_key} <- {table_cfg['file']}")

    raw_df = read_raw_csv(spark, file_path)

    clean_df = prepare_table(
        raw_df,
        col_rename=rename_map(table_key),
        col_types=type_map(table_key),
    )

    write_to_mysql(
        clean_df,
        table_cfg["db_table"],
        cfg["jdbc_url"],
        cfg["db_user"],
        cfg["db_password"],
    )

    print(f"Done: {table_key}")

In [ ]:
spark.stop()